In [6]:
import requests
import json
import pandas as pd
from datetime import datetime
from collections import defaultdict, Counter

# APIキーを設定
API_KEY = "jyxnn12o9ar6qzx4mth0siy822h76i6uxa6was38unmvid0f6q3nuoszd11ulcmj"

### ノード情報の取得と構築
駅名, 路線コードリスト[], 路線リスト[], 緯度, 経度, 1日の平均乗降者数(最新年度)

In [24]:
station_url = "https://api.odpt.org/api/v4/odpt:Station" # 駅情報API
survey_url = "https://api.odpt.org/api/v4/odpt:PassengerSurvey" # 乗降者数API

# 事業者と路線を指定
operators = [
    ("odpt.Operator:TokyoMetro", "TokyoMetro.", None),
    ("odpt.Operator:Toei", "Toei.", {"Toei.Asakusa", "Toei.Mita", "Toei.Oedo", "Toei.Shinjuku"})
]

# URIから末尾ID（例：TokyoMetro.Ginza.Ueno）を抽出する関数
def get_station_key(uri):
    return uri.split(":")[-1]

# 駅＋乗降者数データの統合
station_map = {}
all_years = []

for operator, prefix, railway_filter in operators:
    params = {"odpt:operator": operator, "acl:consumerKey": API_KEY}
    
    station_data = requests.get(station_url, params=params).json()
    survey_data = requests.get(survey_url, params=params).json()

    # 乗降者数マップ
    passenger_map = {}
    for item in survey_data:
        uris = item.get("odpt:station", [])
        for s in item.get("odpt:passengerSurveyObject", []):
            year = s.get("odpt:surveyYear")
            if year:
                all_years.append(year)
            journeys = s.get("odpt:passengerJourneys")
            for uri in uris:
                key = get_station_key(uri)
                passenger_map[key] = journeys

    for s in station_data:
        name = s.get("dc:title")
        same_as = s.get("owl:sameAs", "")
        key = get_station_key(same_as)
        base_code = s.get("odpt:stationCode", "")
        railways = s.get("odpt:railway")
        lat = s.get("geo:lat")
        lon = s.get("geo:long")
        journeys = passenger_map.get(key, 0)

        if isinstance(railways, str):
            railways = [railways]

        lines = []
        for r in railways:
            r_key = r.split(":")[-1]
            if railway_filter is None or r_key in railway_filter:
                lines.append(r_key)

        if not lines:
            continue  # 指定外の路線はスキップ

        # 駅コードの生成（東京メトロ＋都営対応）
        codes = []
        if base_code:
            try:
                codes = []
                if base_code:
                    if "-" in base_code:
                        # 例：SA-09 → A09（Toei.Asakusa → A09）
                        suffix = base_code.split("-")[-1]
                        codes = [line.split(".")[-1][0] + suffix for line in lines]
                    else:
                        # 例：C19 → C19（TokyoMetro.Chiyoda）
                        codes = [base_code]  # そのままでよい
            except:
                codes = []

        if name not in station_map:
            station_map[name] = {
                "id": name,
                "codes": codes,
                "lines": lines,
                "lat": lat,
                "lon": lon,
                "passengers": journeys
            }
        else:
            station_map[name]["codes"].extend(codes)
            station_map[name]["lines"].extend(lines)
            station_map[name]["codes"] = list(set(station_map[name]["codes"]))
            station_map[name]["lines"] = list(set(station_map[name]["lines"]))
            station_map[name]["passengers"] += journeys

# 最新年度を取得
latest_year = max(all_years)

# 出力
nodes = list(station_map.values())
output_data = {
    "metadata": {
        "generated_at": datetime.now().isoformat(),
        "passenger_survey_year": latest_year,
        "included_operators": [op[0] for op in operators],
        "filtered_railways": {
            op[0]: ("All" if op[2] is None else list(op[2])) for op in operators
        }
    },
    "nodes": nodes
}

In [25]:
# JSONで保存
with open("Node_metro_toei.json", "w", encoding="utf-8") as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

### エッジ情報の取得と構築
- 駅名A(始点), 駅名B(終点), 路線名, 路線カラー, 平均移動時間, 頻度, 出発時間[]
    - ローカルのみ（快速等は除外）
    - 上り下り区別可

In [19]:
TARGET_HOUR = 4 # 時間帯指定
TARGET_CALENDAR = "Weekday" # 曜日指定

In [20]:
station_url = "https://api.odpt.org/api/v4/odpt:Station" # 駅情報API
railway_url = "https://api.odpt.org/api/v4/odpt:Railway" # 鉄道路線情報API

operators = [
    ("odpt.Operator:TokyoMetro", None),
    ("odpt.Operator:Toei", {"Toei.Asakusa", "Toei.Mita", "Toei.Oedo", "Toei.Shinjuku"})
]

# 駅・路線・カラー辞書を構築
station_id_to_name = {}
railway_map = {}
all_edges = []
railway_issue_dates = {}

for operator, railway_filter in operators:
    # 駅情報取得
    station_data = requests.get(station_url, params={
        "odpt:operator": operator, "acl:consumerKey": API_KEY
    }).json()

    for s in station_data:
        key = s["owl:sameAs"].split(":")[-1]
        station_id_to_name[key] = s["dc:title"]

    # 路線情報取得
    railway_data = requests.get(railway_url, params={
        "odpt:operator": operator, "acl:consumerKey": API_KEY
    }).json()

    for r in railway_data:
        same_as = r["owl:sameAs"]
        r_key = same_as.split(":")[-1]
        if railway_filter is None or r_key in railway_filter:
            railway_map[same_as] = {
                "name": r_key.split(".")[-1],  # Ginza, Oedo, etc.
                "color": r.get("odpt:color")
            }

# 駅名を取得する関数
def extract_station_name(uri):
    if not uri:
        return None
    return station_id_to_name.get(uri.split(":")[-1], uri)

# 駅間エッジ生成
for railway_uri, info in railway_map.items():
    # 事業者自動判別
    operator = "odpt.Operator:TokyoMetro" if "TokyoMetro" in railway_uri else "odpt.Operator:Toei"

    # TrainTimetable取得
    timetables = requests.get("https://api.odpt.org/api/v4/odpt:TrainTimetable", params={
        "odpt:operator": operator,
        "odpt:railway": railway_uri,
        "odpt:calendar": f"odpt.Calendar:{TARGET_CALENDAR}",
        "acl:consumerKey": API_KEY
    }).json()

    # 日付記録
    issue_list = [t.get("dct:issued") or t.get("dc:date") for t in timetables if t.get("dct:issued") or t.get("dc:date")]
    if issue_list:
        railway_issue_dates[info["name"]] = {
            "latest_issued": max(issue_list),
            "earliest_issued": min(issue_list)
        }

    # 駅間計算
    from collections import defaultdict
    travel_time_dict = defaultdict(list)
    departure_time_dict = defaultdict(list)

    for train in timetables:
        # Local列車のみ
        if "TokyoMetro" in operator and train.get("odpt:trainType") != "odpt.TrainType:TokyoMetro.Local":
            continue
        if "Toei" in operator and train.get("odpt:trainType") not in (None, "odpt.TrainType:Toei.Local"):
            continue

        stops = train.get("odpt:trainTimetableObject", [])
        departures = [
            (extract_station_name(s.get("odpt:departureStation")), s.get("odpt:departureTime"))
            for s in stops
            if "odpt:departureStation" in s and "odpt:departureTime" in s
        ]
        if len(departures) < 2:
            continue

        # 出発→出発による移動時間計算
        for i in range(len(departures) - 1):
            station_a, time_a = departures[i]
            station_b, time_b = departures[i + 1]
            try:
                dt1 = datetime.strptime(time_a, "%H:%M")
                dt2 = datetime.strptime(time_b, "%H:%M")
                if dt1.hour == TARGET_HOUR:
                    diff = (dt2 - dt1).seconds // 60
                    if 0 < diff <= 10:
                        key = (station_a, station_b)
                        travel_time_dict[key].append(diff)
                        departure_time_dict[key].append(time_a)
            except:
                continue

        # 最後の駅の到着処理
        if "odpt:arrivalTime" in stops[-1] and "odpt:arrivalStation" in stops[-1]:
            station_a, time_a = departures[-1]
            station_b = extract_station_name(stops[-1]["odpt:arrivalStation"])
            time_b = stops[-1]["odpt:arrivalTime"]
            try:
                dt1 = datetime.strptime(time_a, "%H:%M")
                dt2 = datetime.strptime(time_b, "%H:%M")
                if dt1.hour == TARGET_HOUR:
                    diff = (dt2 - dt1).seconds // 60
                    if 0 < diff <= 10:
                        key = (station_a, station_b)
                        travel_time_dict[key].append(diff)
                        departure_time_dict[key].append(time_a)
            except:
                continue

    # エッジ記録
    for (station_a, station_b), times in travel_time_dict.items():
        if not times:
            continue
        avg = round(sum(times) / len(times), 1)
        departure_times = sorted(
            departure_time_dict[(station_a, station_b)],
            key=lambda t: datetime.strptime(t, "%H:%M")
        )
        all_edges.append({
            "station_a": station_a,
            "station_b": station_b,
            "line": info["name"],
            "line_color": info["color"],
            "average_time": avg,
            "count": len(times),
            "departure_times": departure_times
        })

# 出力
output_data = {
    "metadata": {
        "generated_at": datetime.now().isoformat(),
        "target_hour": TARGET_HOUR,
        "calendar": TARGET_CALENDAR,
        "railway_issued_dates": railway_issue_dates
    },
    "edges": all_edges
}

In [30]:
# JSONで保存
with open("Edge_metro_toei.json", "w", encoding="utf-8") as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

### ノートとエッジ情報の簡易表示

In [32]:
import json
import pandas as pd
import networkx as nx
import plotly.graph_objs as go

# データ読み込み
with open("Node_metro_toei.json", encoding="utf-8") as f:
    node_data = json.load(f)
nodes = node_data["nodes"]

with open("Edge_metro_toei.json", encoding="utf-8") as f:
    edge_data = json.load(f)
edges = edge_data["edges"]

# グラフ構築
G = nx.DiGraph()

# ノード追加
for node in nodes:
    G.add_node(
        node["id"],
        lat=node["lat"],
        lon=node["lon"],
        passengers=node["passengers"]
    )

# エッジ追加
for edge in edges:
    G.add_edge(
        edge["station_a"],
        edge["station_b"],
        line=edge["line"],
        color=edge["line_color"],
        avg_time=edge["average_time"],
        count=edge["count"]
    )

# 座標抽出
lats = [G.nodes[n]["lat"] for n in G.nodes]
lons = [G.nodes[n]["lon"] for n in G.nodes]
labels = list(G.nodes)

# ノード（駅）のプロット
node_trace = go.Scattermapbox(
    lat=lats,
    lon=lons,
    mode='markers+text',
    marker=dict(size=6, color='blue'),
    text=labels,
    textposition="top center"
)

# エッジ（路線）のプロット
edge_traces = []
for u, v, d in G.edges(data=True):
    # 適切なスケーリング（例: 1〜6の範囲に正規化）
    width = max(1, min(d["count"] / 10, 6))  # 10本で幅1、60本で幅6
    edge_traces.append(go.Scattermapbox(
        lat=[G.nodes[u]["lat"], G.nodes[v]["lat"]],
        lon=[G.nodes[u]["lon"], G.nodes[v]["lon"]],
        mode='lines',
        line=dict(width=width, color=d["color"] or "#888"),
        hoverinfo="text",
        text=f'{u} → {v}<br>{d["line"]}線<br>{d["avg_time"]}分<br>{d["count"]}本'
    ))

# プロット合成
fig = go.Figure(data=[*edge_traces, node_trace])
fig.update_layout(
    mapbox_style="carto-positron",
    mapbox_zoom=10,
    mapbox_center={"lat": 35.68, "lon": 139.76},
    margin={"r":0, "t":0, "l":0, "b":0}
)

fig.show()